# Results for model: nvidia/llama-3.3-nemotron-super-49b-v1

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Load Data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Ensure date_id is datetime
df = df.with_columns([pl.col('date_id').cast(pl.Datetime)])


# Feature Engineering
def calculate_top_quantile(df):
    top_quantile = 0.15  # Top 15%
    df = df.with_columns([
        pl.col('feature_00').ewm(span=7, adjust=False).mean().alias('feature_00_7day_mean'),
        pl.col('feature_00').ewm(span=14, adjust=False).mean().alias('feature_00_14day_mean'),
        pl.col('feature_00').ewm(span=30, adjust=False).mean().alias('feature_00_30day_mean'),
        pl.col('feature_00').rolling_window('date_id', '7D').mean().alias('feature_00_7day_roll_mean'),
        (pl.col('feature_00') 
         .rolling_window('date_id', '7D')
         .quantile(top_quantile)  # Calculate top 15% quantile
         .shift(1)  # Shift to avoid lookahead
         .alias('TOP_QUANTILE')),
        (pl.col('feature_00') 
         .rolling_window('date_id', '14D')
         .quantile(top_quantile)  
         .shift(1) 
         .alias('TOP_QUANTILE_14D')),
        pl.col('feature_00').rolling_window('date_id', '30D').quantile(top_quantile).shift(1).alias('TOP_QUANTILE_30D')
    ])
    return df


df = calculate_top_quantile(df)

# Prepare for Modeling
X = df.drop(['date_id', 'responder_6']).to_pandas()
y = df['responder_6'].to_pandas()

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost Regressor
model = xgb.XGBRegressor(objective='reg:squarederror', 
                         max_depth=7, 
                         learning_rate=0.05, 
                         n_estimators=1000, 
                         n_jobs=-1)
model.fit(X_train, y_train)

# Predict and Evaluate
y_pred = model.predict(X_test)
print(f'MSE: {mean_squared_error(y_test, y_pred)}')